# 04. Dict, set, and generator expressions

**Tier:** Intermediate. **Why you'd use this:** From a list of words you want a dict of word to length, a set of those lengths, and a lazy stream of the long words only. Same comprehension idea, three shapes.

**After this lesson you can:** build a dict or set from a loop in one line; write a lazy sequence with parentheses; choose `[]` / `{}` versus `()` when you care whether work runs now or later.
About 25 minutes. **Uses:** [Comprehensions](01-comprehensions-lesson.ipynb), [Sorting](03-sorting-lesson.ipynb), [Generators](02-generators-lesson.ipynb)


## Quick review

1. Why sort by the secondary key before the primary key?
2. What does `key=len` compare?

<details><summary>Answers</summary>

Stability keeps the secondary order among ties. It compares the lengths, not the strings themselves.
</details>


## The idea

`{key: value for ...}` builds a dict. `{value for ...}` builds a set. `(value for ...)` is a generator expression: lazy like a generator function, written inline. Square brackets always mean a list. Parentheses around a comprehension mean a generator, except when they are just grouping a function call's argument.

<details><summary>Explain more</summary>

A dict comprehension overwrites earlier keys that collide. A set comprehension keeps each value once. A generator expression does not allocate the results until you pull them, and it is single-pass like any generator.
</details>


## See it work

Index words by length, collect unique lengths, and stream the long words.


In [1]:
def summarize(words):
    # 1. Last word of each length wins in the dict.
    by_len = {len(word): word for word in words}
    # 2. Unique lengths.
    lengths = {len(word) for word in words}
    # 3. Lazy stream of long words.
    long = (word for word in words if len(word) >= 4)
    return by_len, lengths, long


by_len, lengths, long = summarize(["aa", "bbbb", "cc", "dddd"])
print(by_len)
print(lengths)
print(list(long))
print(list(long))


{2: 'cc', 4: 'dddd'}
{2, 4}
['bbbb', 'dddd']
[]


## Check yourself

Answer these before you run the cell.

1. Why is the dict's value for key `2` `"cc"` and not `"aa"`?
2. Is `sum(n * n for n in range(4))` a tuple?


In [2]:
print({len(word): word for word in ["aa", "bbbb", "cc"]})
print(type(n * n for n in range(4)).__name__)
print(sum(n * n for n in range(4)))


{2: 'cc', 4: 'bbbb'}
generator
14


<details><summary>Answers</summary>

Later keys overwrite earlier ones, so length `2` ends on `"cc"`. The `sum(...)` argument is a generator expression, not a tuple. `type(...).__name__` is `generator`.
</details>


## Why it works this way

Dict and set comprehensions use the same nested scope rule as list comprehensions. The shape of the brackets chooses the container. A dict needs a key and a value. Duplicate keys are ordinary dict writes: the last write wins. That is why indexing by length keeps only one word per length. A set drops duplicates by hash, so the values must be hashable. Unhashable values raise at build time. Deep dive: [How dict and set work](../../internals/04-dicts/01-hash-table-lesson.ipynb).

A generator expression compiles like a generator function with one `yield` in a loop. Parentheses can be dropped when it is the only argument to a call, which is why `sum(n * n for n in range(4))` works and is not a tuple. The commas that make a tuple are missing, so the parentheses are only grouping. The object is still single-pass and small until you pull. Calling `list` on it spends it. A second consumer that expects the same values must build a fresh expression or keep a list.

Cost follows the container: a dict or set allocates entries as it goes, with hash-table overhead. A generator expression allocates the generator object only. Choose the shape that matches the question you ask next. Need lookup by key? Build a dict. Need membership of values with no order? Build a set. Need one walk and then discard? Use a generator expression. Need two walks or random access? Use a list comprehension.

The design is one syntax family for three containers plus laziness. That keeps reading left to right the same skill you already have from list comprehensions. The brackets are the signal. Square means materialize a list. Curly means a dict or set. Round means lazy. Misreading round as a tuple is the common trap because both use parentheses in other places.


In [3]:
import sys

words = ["aa", "bbbb", "cc", "dddd"]
print({len(word): word for word in words})
print({len(word) for word in words})
gen = (word for word in words if len(word) >= 4)
print(sys.getsizeof(gen) < sys.getsizeof([word for word in words if len(word) >= 4]))
print(list(gen))
print(list(gen))


{2: 'cc', 4: 'dddd'}
{2, 4}
False
['bbbb', 'dddd']
[]


## When to use it

Use a dict comprehension to index. Use a set comprehension for unique values. Use a generator expression for a single lazy walk, especially as a call argument. Use a list comprehension when you need the results twice. Don't use a generator expression when you need `len` or reuse.


## Common mistake

Treating a generator expression like a tuple because of the parentheses, then unpacking it twice. The second unpack is empty.


In [4]:
stream = (word for word in ["aa", "bbbb"] if len(word) >= 2)
first = list(stream)
second = list(stream)
print(first)
print(second)


['aa', 'bbbb']
[]


Build a list if you need two walks, or write the generator expression twice.


In [5]:
words = ["aa", "bbbb"]
first = [word for word in words if len(word) >= 2]
second = [word for word in words if len(word) >= 2]
print(first)
print(second)


['aa', 'bbbb']
['aa', 'bbbb']


## Try it

Change `summarize` so the dict maps each word to its length instead of length to word. What happens to collisions, and which container would you pick if you only needed the lengths?


## Recap

- `{k: v for ...}` builds a dict. `{v for ...}` builds a set.
- `(v for ...)` is a lazy, single-pass generator expression.
- Last key wins in a dict comprehension. Sets need hashable values.

**Next:** [practice](04-more-comprehensions-practice.ipynb)
